To run your Flask app on ports `443` (HTTPS) or `80` (HTTP) on an EC2 Ubuntu instance, you need to set up some additional configurations. By default, Flask runs on port `5000`, but you’ll need to:

1. **Install a web server like Nginx or Apache.**
2. **Redirect traffic from ports `80` and `443` to your Flask app.**
3. **Set up proper permissions for the ports (since only root can bind to ports below 1024).**

Here's a step-by-step guide to set up your Flask app on ports `80` or `443`:

### 1. Install Nginx
Nginx will act as a reverse proxy server, redirecting HTTP/HTTPS traffic to your Flask application running on port `5000`.

#### Commands to install Nginx:

```bash
sudo apt update
sudo apt install nginx
```

### 2. Configure Nginx to Proxy Requests

After installing Nginx, you need to create a configuration file to forward requests from ports `80` or `443` to your Flask app.

#### Create a new Nginx configuration file:

```bash
sudo nano /etc/nginx/sites-available/myflaskapp
```

Add the following content to the file (this is for HTTP on port 80):

```nginx
server {
    listen 80;
    server_name YOUR_PUBLIC_IP;  # Replace YOUR_PUBLIC_IP with your EC2 Public IP

    location / {
        proxy_pass http://127.0.0.1:5000;  # Flask app runs on localhost:5000
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
```

For HTTPS (port 443), you need SSL certificates. Here's a basic example (make sure you replace the SSL certificate and key paths with your own):

```nginx
server {
    listen 443 ssl;
    server_name YOUR_PUBLIC_IP;

    ssl_certificate /path/to/cert.pem;
    ssl_certificate_key /path/to/private.key;

    location / {
        proxy_pass http://127.0.0.1:5000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }
}
```

### 3. Enable the Nginx Site Configuration

Create a symbolic link to enable the site configuration you just created:

```bash
sudo ln -s /etc/nginx/sites-available/myflaskapp /etc/nginx/sites-enabled/
```

### 4. Test Nginx Configuration

Before restarting Nginx, it's always a good idea to test the configuration for errors:

```bash
sudo nginx -t
```

If you get an OK message, you're good to go!

### 5. Restart Nginx

Restart Nginx to apply the changes:

```bash
sudo systemctl restart nginx
```

### 6. Allow Traffic on Ports 80 and 443 in Security Group

1. Go to the **AWS EC2 Console**.
2. Select your instance.
3. Click on the **Security Group**.
4. Add inbound rules for ports `80` and `443` to allow HTTP and HTTPS traffic.

### 7. Update Flask to Run on Port 5000 (Optional)

Ensure your Flask app is running on `localhost:5000`, and it's ready to handle incoming requests forwarded by Nginx.

In your Flask app, make sure you run it on port `5000` (the default port) and set it to accept connections only from localhost (127.0.0.1):

```python
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000)
```

### 8. (Optional) Set Up Flask for Production (Using Gunicorn)

It's recommended to run Flask in production with a WSGI server like **Gunicorn** instead of the Flask built-in server. Here’s how to install and use it:

#### Install Gunicorn:

```bash
pip install gunicorn
```

#### Run your Flask app with Gunicorn:

```bash
gunicorn --bind 127.0.0.1:5000 wsgi:app  # Assuming wsgi.py is where your Flask app is defined
```

Now, your Flask app will be ready to handle production traffic.

---

### Summary:

1. Install Nginx.
2. Configure Nginx to forward requests from ports `80` or `443` to `5000` (Flask's default port).
3. Test and restart Nginx.
4. Ensure that security groups in AWS allow traffic on ports `80` and `443`.
5. Optionally, use Gunicorn to run your Flask app.

After this setup, your Flask app should be accessible via HTTP (port 80) or HTTPS (port 443) on your EC2 instance's public IP.
